# Optativa III: Ciencia de Datos
## Revisión técnica avanzada — Transformación de datos e integración total sobre Amazon

**Universidad de Especialidades UNE · Plantel Centro**
**Ingeniería en Computación · 9.º semestre · Semana 4 · Actividad integradora avanzada**

---

### Propósito

Esta actividad de **nivel avanzado** integra la **transformación de datos** (Semana 4) con todo el análisis estadístico del curso (Semanas 2 y 3), sobre el dataset real de **Amazon**. Aquí demostrarás que dominas el flujo profesional completo: **transformar correctamente los datos → analizarlos → modelarlos → interpretar el efecto de cada transformación en los resultados**.

El foco está en algo que los principiantes ignoran: **las transformaciones no son cosméticas, cambian lo que un modelo puede aprender e interpretar**. Verás, por ejemplo, por qué estandarizar permite comparar la importancia de las variables, o por qué codificar mal una categoría arruina un análisis.

Responderás **50 preguntas ✍️** de análisis y de modificación de código, con complejidad alta.

### El dataset

`amazon.csv` — Amazon Sales Dataset (1,465 productos limpios: category, discounted_price, actual_price, discount_percentage, rating, rating_count). Súbelo a Colab antes de empezar.

### Cómo trabajar

Ejecuta cada celda en orden con **Shift + Enter**, responde las preguntas ✍️, completa los retos y entrega en Classroom. El nivel es alto: prioriza la interpretación sobre la ejecución mecánica.


---
# FASE 1 — Transformaciones fundamentales

## Paso 0 — Carga y perfil de variables


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.linear_model import LinearRegression

pd.set_option("display.max_columns", None)
np.random.seed(42)

df = pd.read_csv("amazon.csv")
print("Dimensiones:", df.shape)
print("\nTipos:")
print(df.dtypes)
df.head()

> ✍️ **1.** Clasifica cada columna del dataset: ¿cuáles son numéricas continuas, cuáles categóricas nominales y cuáles podrían tratarse como ordinales? Justifica.

> ✍️ **2.** Las variables `actual_price` (miles) y `rating` (1-5) están en escalas radicalmente distintas. Menciona dos tipos de análisis o algoritmos donde esta diferencia de escala causaría problemas si no la corrigiéramos.

> ✍️ **3.** Antes de transformar, ¿por qué es importante conocer la **distribución** de cada variable? (Pista: la transformación adecuada depende de si la variable es sesgada, tiene outliers, etc.)


## Paso 1 — Normalización vs estandarización: comparación técnica


In [ ]:
# Aplicamos AMBAS transformaciones a actual_price para compararlas
df["precio_norm"] = MinMaxScaler().fit_transform(df[["actual_price"]])
df["precio_z"] = StandardScaler().fit_transform(df[["actual_price"]])

print("ORIGINAL   -> min: {:.0f}  max: {:.0f}  media: {:.0f}".format(
    df["actual_price"].min(), df["actual_price"].max(), df["actual_price"].mean()))
print("NORMALIZADO-> min: {:.3f}  max: {:.3f}  media: {:.3f}".format(
    df["precio_norm"].min(), df["precio_norm"].max(), df["precio_norm"].mean()))
print("ESTANDARIZ.-> min: {:.3f}  max: {:.3f}  media: {:.3f}".format(
    df["precio_z"].min(), df["precio_z"].max(), df["precio_z"].mean()))

fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].hist(df["actual_price"], bins=40, color="#888"); axes[0].set_title("Original")
axes[1].hist(df["precio_norm"], bins=40, color="#1e2a6b"); axes[1].set_title("Min-Max [0,1]")
axes[2].hist(df["precio_z"], bins=40, color="#e2231a"); axes[2].set_title("Z-score")
plt.tight_layout(); plt.show()

> ✍️ **4.** Compara los tres histogramas. ¿La **forma** de la distribución cambió con alguna transformación, o solo cambió la escala del eje? ¿Qué implica esto?

> ✍️ **5.** La media del salario normalizado NO es 0.5, sino menor. ¿Por qué? (Pista: la distribución de precios está sesgada a la derecha, y Min-Max no corrige el sesgo.)

> ✍️ **6.** Ni la normalización ni la estandarización corrigen el **sesgo** de una distribución. Investiga: ¿qué transformación SÍ reduce el sesgo de una variable como el precio? (Pista: logaritmo.)


### 🔧 Reto 1 — Transformación logarítmica

Crea `precio_log` aplicando `np.log1p(actual_price)` (logaritmo). Grafica su histograma y compáralo con el original. ¿Se ve más simétrico?


In [ ]:
# RETO: transformacion logaritmica del precio
# df["precio_log"] = np.log1p(df["actual_price"])
# plt.hist(df["precio_log"], bins=40, color="#1e2a6b")
# plt.title("Precio (escala log)"); plt.show()


> ✍️ **7.** ¿La transformación logarítmica hizo la distribución del precio más simétrica? ¿Por qué en ciencia de datos se aplica log a variables como precios, ingresos o poblaciones?

> ✍️ **8.** ¿Cuándo preferirías estandarización (Z-score) sobre normalización (Min-Max)? Da un criterio claro basado en la presencia de outliers.


## Paso 2 — Codificación de la categoría


In [ ]:
print("Categorias unicas:", df["category"].nunique())
print(df["category"].value_counts())

# One-hot encoding
dummies = pd.get_dummies(df["category"], prefix="cat")
print("\nColumnas one-hot creadas:", len(dummies.columns))

# Label encoding (para comparar)
df["category_label"] = LabelEncoder().fit_transform(df["category"])
print("\nLabel encoding (primeros mapeos):")
print(df[["category", "category_label"]].drop_duplicates().head())

> ✍️ **9.** ¿Cuántas categorías hay y cuántas columnas creó el one-hot? ¿Es manejable ese número de columnas?

> ✍️ **10.** El label encoding asignó números 0,1,2... a las categorías. ¿Por qué sería un ERROR usar `category_label` directamente en una regresión lineal? (Pista: introduce un orden falso.)

> ✍️ **11.** Explica la diferencia práctica: si un modelo usa `category_label`, trataría la categoría 4 como "el doble" de la categoría 2. Con one-hot, ¿cómo se evita ese problema?


### 🔧 Reto 2 — Frequency encoding

Una técnica avanzada es el **frequency encoding**: reemplazar cada categoría por su frecuencia. Créalo: `df["cat_freq"] = df["category"].map(df["category"].value_counts(normalize=True))`.


In [ ]:
# RETO: frequency encoding de category
# df["cat_freq"] = df["category"].map(df["category"].value_counts(normalize=True))
# print(df[["category", "cat_freq"]].drop_duplicates().head())


> ✍️ **12.** ¿Qué representa el valor de `cat_freq` para cada categoría? ¿En qué situación el frequency encoding sería preferible al one-hot? (Pista: muchas categorías.)


## Paso 3 — Feature engineering avanzado


In [ ]:
# Feature 1: ahorro absoluto y relativo
df["ahorro"] = df["actual_price"] - df["discounted_price"]

# Feature 2: ratio precio/popularidad (precio por cada mil resenas)
df["precio_por_popularidad"] = (df["actual_price"] / (df["rating_count"]/1000 + 1)).round(2)

# Feature 3: indicador de "oferta fuerte"
df["oferta_fuerte"] = (df["discount_percentage"] > 50).astype(int)

# Feature 4: score de valor (rating alto + precio bajo = buena compra)
df["score_valor"] = (df["rating"] / (df["actual_price"]/1000 + 1)).round(3)

print(df[["ahorro", "precio_por_popularidad", "oferta_fuerte", "score_valor"]].describe())

> ✍️ **13.** Explica qué mide cada una de las cuatro features creadas. ¿Cuál combina más variables?

> ✍️ **14.** La feature `score_valor` intenta capturar "buena compra" (rating alto, precio bajo). ¿Es una feature válida o artificial? Argumenta y propón cómo la validarías.

> ✍️ **15.** ¿Cuántos productos tienen `oferta_fuerte = 1`? ¿Qué proporción del catálogo está con más de 50% de descuento?

> ✍️ **16.** El feature engineering requiere criterio: una feature mal diseñada añade ruido. ¿Cómo decidirías si una feature nueva realmente aporta valor a un modelo? (Pista: correlación con el objetivo, importancia en el modelo.)


### 🔧 Reto 3 — Tu feature con conocimiento de dominio

Crea una feature `precio_alto_mal_valorado`: productos con precio en el cuartil superior (top 25%) PERO rating por debajo de la mediana. Estos son "caros pero malos". ¿Cuántos hay?


In [ ]:
# RETO: identifica productos caros y mal valorados
# umbral_precio = df["actual_price"].quantile(0.75)
# umbral_rating = df["rating"].median()
# df["precio_alto_mal_valorado"] = ((df["actual_price"] >= umbral_precio) &
#                                   (df["rating"] < umbral_rating)).astype(int)
# print("Productos caros y mal valorados:", df["precio_alto_mal_valorado"].sum())


> ✍️ **17.** ¿Cuántos productos son "caros pero mal valorados"? ¿Por qué esta feature podría ser valiosa para un comprador o para Amazon (detectar productos problemáticos)?


## Paso 4 — Binning (discretización)


In [ ]:
# Convertimos el precio continuo en categorias (cuartiles)
df["precio_categoria"] = pd.qcut(df["actual_price"], q=4,
                                 labels=["Económico", "Medio", "Caro", "Premium"])

print("Distribucion por categoria de precio:")
print(df["precio_categoria"].value_counts())

print("\nRating promedio por categoria de precio:")
print(df.groupby("precio_categoria", observed=True)["rating"].mean().round(3))

> ✍️ **18.** ¿Qué hace `pd.qcut()` y en qué se diferencia de `pd.cut()`? (Pista: uno usa cuartiles, el otro rangos fijos.)

> ✍️ **19.** Mira el rating promedio por categoría de precio. ¿Los productos "Premium" tienen mejor rating que los "Económicos"? ¿Qué confirma esto sobre la relación precio-calidad?

> ✍️ **20.** ¿Qué se GANA y qué se PIERDE al convertir el precio continuo en 4 categorías (binning)? Menciona una ventaja y una desventaja.


---
# FASE 2 — Efecto de las transformaciones en el análisis

## Paso 5 — Estandarización y comparación de coeficientes


In [ ]:
# Regresion para predecir discounted_price con 3 variables
predictoras = ["actual_price", "discount_percentage", "rating_count"]
y = df["discounted_price"]

# Modelo SIN estandarizar
m_crudo = LinearRegression().fit(df[predictoras], y)
print("Coeficientes SIN estandarizar (NO comparables entre si):")
for v, c in zip(predictoras, m_crudo.coef_):
    print(f"  {v}: {c:.4f}")
print(f"  R2: {m_crudo.score(df[predictoras], y):.4f}")

# Modelo CON estandarizacion
X_z = StandardScaler().fit_transform(df[predictoras])
m_z = LinearRegression().fit(X_z, y)
print("\nCoeficientes CON estandarizacion (SI comparables entre si):")
for v, c in zip(predictoras, m_z.coef_):
    print(f"  {v}: {c:.2f}")
print(f"  R2: {m_z.score(X_z, y):.4f}")

> ✍️ **21.** El R² es idéntico con y sin estandarizar. ¿Por qué la estandarización NO cambia el poder predictivo del modelo?

> ✍️ **22.** Aunque el R² no cambia, los coeficientes SÍ. Con estandarización, ¿cuál de las tres variables tiene el mayor impacto (coeficiente absoluto más grande) sobre el precio con descuento? ¿Podrías haberlo sabido con los coeficientes crudos?

> ✍️ **23.** Explica por qué los coeficientes crudos NO son comparables entre sí, pero los estandarizados SÍ. (Pista: cada variable cruda está en unidades distintas.)

> ✍️ **24.** Esta es una lección clave: la estandarización no mejora la predicción, pero sí la **interpretabilidad**. ¿En qué situación de tu trabajo como analista necesitarías comparar la importancia de variables en distintas unidades?


### 🔧 Reto 4 — Importancia de variables

Ordena las tres variables predictoras por la magnitud (valor absoluto) de su coeficiente estandarizado, de mayor a menor impacto. Crea un gráfico de barras.


In [ ]:
# RETO: grafica la importancia (coef estandarizado absoluto)
# importancia = pd.Series(np.abs(m_z.coef_), index=predictoras).sort_values(ascending=False)
# importancia.plot(kind="barh", color="#1e2a6b")
# plt.title("Importancia de variables (coef estandarizado)"); plt.show()


> ✍️ **25.** Según tu gráfico, ordena las variables de más a menos importante para predecir el precio con descuento. ¿El resultado tiene sentido lógico?


## Paso 6 — Correlación con features transformadas


In [ ]:
# Matriz de correlacion incluyendo features nuevas
cols_analisis = ["actual_price", "discount_percentage", "rating", "rating_count",
                 "ahorro", "score_valor", "oferta_fuerte"]
matriz = df[cols_analisis].corr()

plt.figure(figsize=(9,7))
sns.heatmap(matriz, annot=True, cmap="coolwarm", center=0, fmt=".2f", square=True, linewidths=1)
plt.title("Correlacion (variables originales + features nuevas)")
plt.tight_layout(); plt.show()

> ✍️ **26.** ¿Cuál de tus features nuevas tiene la correlación más fuerte con alguna variable original? ¿Tiene sentido esa correlación?

> ✍️ **27.** La feature `ahorro` probablemente correlaciona fuerte con `actual_price`. ¿Por qué? ¿Sería redundante incluir ambas en un modelo? (Pista: multicolinealidad.)

> ✍️ **28.** ¿Una feature que correlaciona casi perfectamente con una variable existente aporta información nueva? ¿Cuándo el feature engineering se vuelve redundante?


---
# FASE 3 — Pipeline reproducible completo

## Paso 7 — Pipeline integral


In [ ]:
def pipeline_completo(datos):
    """Pipeline de transformacion reproducible para el dataset de Amazon.
    Recibe datos crudos y devuelve datos transformados listos para modelar."""
    d = datos.copy()

    # 1. Features derivadas
    d["ahorro"] = d["actual_price"] - d["discounted_price"]
    d["oferta_fuerte"] = (d["discount_percentage"] > 50).astype(int)
    d["score_valor"] = (d["rating"] / (d["actual_price"]/1000 + 1)).round(3)

    # 2. Transformacion logaritmica del precio (reduce sesgo)
    d["precio_log"] = np.log1p(d["actual_price"])

    # 3. Estandarizacion de numericas
    for col in ["actual_price", "discount_percentage", "rating_count"]:
        d[col + "_z"] = StandardScaler().fit_transform(d[[col]])

    # 4. One-hot de la categoria
    d = pd.concat([d, pd.get_dummies(d["category"], prefix="cat")], axis=1)

    # 5. Binning del precio
    d["precio_categoria"] = pd.qcut(d["actual_price"], q=4,
                                    labels=["Económico", "Medio", "Caro", "Premium"])
    return d

df_crudo = pd.read_csv("amazon.csv")
df_final = pipeline_completo(df_crudo)
print("Crudo:", df_crudo.shape, "-> Transformado:", df_final.shape)
print("\nColumnas nuevas:", [c for c in df_final.columns if c not in df_crudo.columns])

> ✍️ **29.** ¿Cuántas columnas nuevas agregó el pipeline completo? Enuméralas por tipo de transformación (features, log, estandarización, one-hot, binning).

> ✍️ **30.** ¿Por qué un pipeline como este es esencial para la **reproducibilidad**? Si publicaras un artículo científico con este análisis, ¿cómo ayudaría el pipeline a que otros verifiquen tus resultados?

> ✍️ **31.** El pipeline aplica las transformaciones en un orden específico. ¿Importa el orden? Da un ejemplo de dos pasos donde el orden sí importaría.


### 🔧 Reto 5 — Amplía el pipeline

Agrega al pipeline un paso que cree la feature `precio_alto_mal_valorado` (del Reto 3). Vuelve a ejecutarlo y verifica que aparezca.


In [ ]:
# RETO: copia pipeline_completo, agrega el paso y reejecuta


> ✍️ **32.** Tras ampliar el pipeline, ¿cuántas columnas tiene ahora el dataset final? ¿Por qué un pipeline modular (fácil de extender) es una buena práctica de ingeniería de software aplicada a datos?


## Paso 8 — Data leakage: el error que arruina modelos


In [ ]:
# DEMOSTRACION del error de data leakage al estandarizar
from sklearn.model_selection import train_test_split

X = df[["actual_price"]].values
y = df["discounted_price"].values

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

# CORRECTO: ajustar el scaler SOLO con train, aplicar a ambos
scaler = StandardScaler()
X_train_z = scaler.fit_transform(X_train)   # fit SOLO con train
X_test_z = scaler.transform(X_test)         # solo transform en test

print("Media del scaler (aprendida de train):", round(scaler.mean_[0], 2))
print("X_train_z media:", round(X_train_z.mean(), 4))
print("X_test_z media:", round(X_test_z.mean(), 4), " (NO es exactamente 0: correcto)")

> ✍️ **33.** El **data leakage** (fuga de datos) ocurre cuando información del conjunto de prueba se filtra al entrenamiento. En la estandarización, ¿por qué debemos hacer `fit` SOLO con los datos de entrenamiento?

> ✍️ **34.** ¿Por qué la media de `X_test_z` NO es exactamente 0, y por qué eso es CORRECTO? (Pista: el scaler se ajustó con train, no con test.)

> ✍️ **35.** Si hubiéramos estandarizado TODO el dataset antes de dividir en train/test, ¿qué información se habría filtrado? ¿Por qué inflaría artificialmente el rendimiento del modelo?

> ✍️ **36.** El data leakage es uno de los errores más graves y sutiles en ciencia de datos. Menciona otro escenario (además de la estandarización) donde podría ocurrir una fuga de datos.


---
# FASE 4 — Análisis integrador sobre datos transformados

## Paso 9 — Análisis por segmentos


In [ ]:
# Analisis: caracteristicas promedio por categoria de precio
resumen = df.groupby("precio_categoria", observed=True).agg(
    rating_promedio=("rating", "mean"),
    descuento_promedio=("discount_percentage", "mean"),
    resenas_promedio=("rating_count", "mean"),
    num_productos=("category", "count")
).round(2)
print(resumen)

> ✍️ **37.** Analiza la tabla: ¿los productos Premium reciben más o menos descuento que los Económicos? ¿Tienen más reseñas?

> ✍️ **38.** El rating es casi constante entre las cuatro categorías de precio. Enlaza esto con TODO lo que has visto en el curso: correlación precio-rating, prueba t, regresión. ¿Qué conclusión robusta emerge?

> ✍️ **39.** ¿Cómo te ayudó el binning (`precio_categoria`) a hacer este análisis por segmentos? ¿Podrías haberlo hecho igual de fácil con el precio continuo?


### 🔧 Reto 6 — Análisis cruzado

Crea una tabla que muestre, para cada categoría de producto (`category`), el precio promedio y el rating promedio, ordenada por precio. ¿La categoría más cara es la mejor valorada?


In [ ]:
# RETO: análisis por category
# tabla = df.groupby("category").agg(
#     precio_prom=("actual_price", "mean"),
#     rating_prom=("rating", "mean")
# ).round(2).sort_values("precio_prom", ascending=False)
# print(tabla)


> ✍️ **40.** ¿La categoría de producto más cara es también la mejor valorada? ¿Qué implica esto para la estrategia de precios de un vendedor?


## Paso 10 — Modelo final sobre datos transformados


In [ ]:
# Modelo final: predecir discounted_price con variables transformadas
features_modelo = ["actual_price_z", "discount_percentage_z", "rating_count_z"]
X = df_final[features_modelo]
y = df_final["discounted_price"]

modelo = LinearRegression().fit(X, y)
print("Modelo sobre datos transformados:")
print(f"  R2: {modelo.score(X, y):.4f}")
for v, c in zip(features_modelo, modelo.coef_):
    print(f"  {v}: {c:.2f}")

# Residuos
pred = modelo.predict(X)
residuos = y - pred
plt.figure(figsize=(9,5))
plt.scatter(pred, residuos, alpha=0.3, color="#1e2a6b")
plt.axhline(0, color="#e2231a", linestyle="--")
plt.title("Residuos del modelo (datos transformados)")
plt.xlabel("Predicho"); plt.ylabel("Residuo")
plt.grid(alpha=0.2); plt.show()

> ✍️ **41.** ¿Cuál es el R² del modelo final? Con los coeficientes estandarizados, ¿qué variable domina la predicción?

> ✍️ **42.** Analiza los residuos: ¿se distribuyen de forma pareja alrededor de cero o hay un patrón? ¿Se cumple el supuesto de homocedasticidad?

> ✍️ **43.** Compara este modelo (sobre datos transformados) con lo que habrías obtenido sobre datos crudos: el R² es el mismo, pero ¿qué ganaste al transformar en términos de interpretación?


---
# FASE 5 — Síntesis de alto nivel

Estas preguntas exigen conectar la transformación con todo el flujo de ciencia de datos.


> ✍️ **44.** Recorre el pipeline completo que ejecutaste (limpieza previa → transformación → análisis → modelado). ¿En qué punto exacto encaja la transformación de datos dentro del proceso KDD/CRISP-DM que viste en la Semana 1?

> ✍️ **45.** Un compañero dice: "La transformación de datos es opcional, solo cambia números". Desmiente esta afirmación con TRES ejemplos concretos de esta actividad donde la transformación cambió lo que pudimos analizar o interpretar.

> ✍️ **46.** Ordena, de mayor a menor impacto, estas transformaciones según cuánto afectan la interpretación de un modelo: estandarización, one-hot encoding, binning, transformación logarítmica. Justifica tu orden.

> ✍️ **47.** El feature engineering se considera "más arte que ciencia". Con base en tu experiencia en esta actividad, ¿estás de acuerdo? ¿Qué papel juega el conocimiento del dominio (entender de productos, precios, clientes)?

> ✍️ **48.** Integra los hallazgos del curso: correlación precio-rating ≈ 0, prueba t no significativa, rating constante entre categorías de precio, y el modelo de rating con R² casi nulo. ¿Qué historia única y robusta cuentan TODOS juntos sobre "precio y calidad" en Amazon?

> ✍️ **49.** **Reproducibilidad y ética:** ¿por qué documentar cada transformación (en un pipeline y un diccionario de datos) es una responsabilidad profesional y no solo una buena práctica? Piensa en quién audita o hereda tu trabajo.

> ✍️ **50 (reflexión final).** Has recorrido el flujo completo de la ciencia de datos: fundamentos (S1), estadística (S2), inferencia (S3), limpieza y transformación (S4). Describe un proyecto de tu carrera de Ingeniería en Computación donde aplicarías TODO este flujo de principio a fin, especificando qué transformaciones necesitarías.


---
## Cierre y entrega

Dominaste la transformación de datos a nivel profesional: normalización, estandarización, codificación (one-hot, label, frequency), feature engineering avanzado, binning, transformación logarítmica y, sobre todo, comprendiste **cómo cada transformación afecta el análisis y la interpretación**, además de evitar el error crítico del data leakage. Construiste un **pipeline reproducible** integrando todo el curso.

### Entrega en Google Classroom

1. Verifica que **todas las celdas corran sin error** (Entorno de ejecución → Ejecutar todo).
2. Confirma que respondiste las **50 preguntas ✍️** y los **6 retos de código**.
3. **Archivo → Guardar una copia en Drive** y comparte el enlace en Classroom.

### Rúbrica

| Criterio | Puntos |
|---|---|
| Transformaciones correctas y completas | 20 |
| Retos de código resueltos (6) | 20 |
| Respuestas a las 50 preguntas ✍️ | 35 |
| Comprensión de efectos (estandarización, leakage) | 15 |
| Análisis integrador de alto nivel (Fase 5) | 10 |
| **Total** | **100** |

---
*Universidad de Especialidades UNE · Plantel Centro · Optativa III: Ciencia de Datos*
*Dataset: amazon.csv (Amazon Sales Dataset).*
